In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

CSV_PATH = Path(r"tagged_shots2025.csv")
TARGET_TEAM = "Ingraham"  
TOP_N_PLAYERS = 6
HEATMAP_STAT = "sum"  

# Half-pitch coordinates from your logger
X_MIN, X_MAX = -40, 40
Y_MIN, Y_MAX = 0, 60
GOAL_WIDTH = 8

RESULT_ORDER = ["Goal", "Saved", "Blocked", "Off Target", "Post", "Other"]
RESULT_STYLES = {
    "Goal": {"marker": "*", "color": "gold", "size": 140},
    "Saved": {"marker": "o", "color": "royalblue", "size": 40},
    "Blocked": {"marker": "s", "color": "firebrick", "size": 40},
    "Off Target": {"marker": "x", "color": "gray", "size": 40},
    "Post": {"marker": "D", "color": "darkorange", "size": 45},
    "Other": {"marker": "^", "color": "black", "size": 40},
}


def safe_slug(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(text).strip().lower()).strip("_")



def load_data(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    required = {
        "Team", "GameId", "Player", "Result", "Type", "X", "Y",
        "shot_distance", "shot_angle"
    }
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    for col in ["Team", "GameId", "Player", "Result", "Type"]:
        df[col] = df[col].astype(str).str.strip()

    if "open_play_or_set_piece" not in df.columns:
        df["open_play_or_set_piece"] = "Unknown"
    if "competition_level" not in df.columns:
        df["competition_level"] = "Unknown"

    df["open_play_or_set_piece"] = df["open_play_or_set_piece"].astype(str).str.strip()
    df["goal"] = df["Result"].str.lower().eq("goal").astype(int)
    df["X"] = pd.to_numeric(df["X"], errors="coerce")
    df["Y"] = pd.to_numeric(df["Y"], errors="coerce")
    df["shot_distance"] = pd.to_numeric(df["shot_distance"], errors="coerce")
    df["shot_angle"] = pd.to_numeric(df["shot_angle"], errors="coerce")

    if "xG" in df.columns:
        df["xG"] = pd.to_numeric(df["xG"], errors="coerce")

    df = df.dropna(subset=["X", "Y", "shot_distance", "shot_angle"]).copy()
    df = df[(df["X"] >= X_MIN) & (df["X"] <= X_MAX) & (df["Y"] >= Y_MIN) & (df["Y"] <= Y_MAX)].copy()
    df["ResultClean"] = df["Result"].apply(clean_result)
    return df



def clean_result(value: str) -> str:
    text = str(value).strip().lower()
    if text == "goal":
        return "Goal"
    if "save" in text:
        return "Saved"
    if "block" in text:
        return "Blocked"
    if "off" in text:
        return "Off Target"
    if "post" in text or "bar" in text or "wood" in text:
        return "Post"
    return "Other"



def choose_team(df: pd.DataFrame) -> str:
    teams = sorted(t for t in df["Team"].dropna().unique() if str(t).strip())
    if TARGET_TEAM:
        if TARGET_TEAM not in teams:
            raise ValueError(f"TARGET_TEAM '{TARGET_TEAM}' not found. Available teams: {teams}")
        return TARGET_TEAM

    print("Available teams:")
    for team in teams:
        print(f" - {team}")
    selected = input("\nType the team name to build the report: ").strip()
    if selected not in teams:
        raise ValueError(f"'{selected}' not found. Available teams: {teams}")
    return selected



def fit_local_xg_model(df: pd.DataFrame):
    X = sm.add_constant(df[["shot_distance", "shot_angle"]])
    y = df["goal"]
    model = sm.Logit(y, X)
    result = model.fit(disp=False)
    return result



def add_xg(df: pd.DataFrame):
    out = df.copy()
    if "xG" in out.columns and out["xG"].notna().all():
        return out, None

    result = fit_local_xg_model(out)
    X = sm.add_constant(out[["shot_distance", "shot_angle"]], has_constant="add")
    out["xG"] = result.predict(X)
    return out, result



def draw_half_pitch(ax):
    ax.plot([X_MIN, X_MAX], [Y_MIN, Y_MIN], color="black", linewidth=2)
    ax.plot([X_MIN, X_MIN], [Y_MIN, Y_MAX], color="black", linewidth=2)
    ax.plot([X_MAX, X_MAX], [Y_MIN, Y_MAX], color="black", linewidth=2)
    ax.plot([X_MIN, X_MAX], [Y_MAX, Y_MAX], color="black", linewidth=2)

    ax.plot([-22, -22], [0, 18], color="black", linewidth=2)
    ax.plot([22, 22], [0, 18], color="black", linewidth=2)
    ax.plot([-22, 22], [18, 18], color="black", linewidth=2)

    ax.plot([-10, -10], [0, 6], color="black", linewidth=2)
    ax.plot([10, 10], [0, 6], color="black", linewidth=2)
    ax.plot([-10, 10], [6, 6], color="black", linewidth=2)

    ax.plot([-GOAL_WIDTH / 2, GOAL_WIDTH / 2], [-1, -1], color="black", linewidth=4)
    ax.plot([-GOAL_WIDTH / 2, -GOAL_WIDTH / 2], [-1, 0], color="black", linewidth=2)
    ax.plot([GOAL_WIDTH / 2, GOAL_WIDTH / 2], [-1, 0], color="black", linewidth=2)

    ax.scatter([0], [12], color="black", s=12)

    ax.set_xlim(X_MIN - 2, X_MAX + 2)
    ax.set_ylim(Y_MAX + 2, -2)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])



def compute_grid(df: pd.DataFrame, stat: str = "sum", x_bins: int = 24, y_bins: int = 18):
    x_edges = np.linspace(X_MIN, X_MAX, x_bins + 1)
    y_edges = np.linspace(Y_MIN, Y_MAX, y_bins + 1)

    if df.empty:
        return np.zeros((y_bins, x_bins)), x_edges, y_edges

    if stat == "sum":
        weights = df["xG"].to_numpy()
        grid, _, _ = np.histogram2d(df["Y"], df["X"], bins=[y_edges, x_edges], weights=weights)
    elif stat == "count":
        grid, _, _ = np.histogram2d(df["Y"], df["X"], bins=[y_edges, x_edges])
    elif stat == "mean":
        sums, _, _ = np.histogram2d(df["Y"], df["X"], bins=[y_edges, x_edges], weights=df["xG"].to_numpy())
        counts, _, _ = np.histogram2d(df["Y"], df["X"], bins=[y_edges, x_edges])
        with np.errstate(divide="ignore", invalid="ignore"):
            grid = np.divide(sums, counts, out=np.zeros_like(sums), where=counts > 0)
    else:
        raise ValueError("stat must be 'sum', 'mean', or 'count'")
    return grid, x_edges, y_edges



def plot_heatmap(df: pd.DataFrame, title: str, filename: Path, stat: str = "sum"):
    grid, x_edges, y_edges = compute_grid(df, stat=stat)
    vmax = float(grid.max()) if np.isfinite(grid.max()) else 1.0
    if vmax <= 0:
        vmax = 1.0

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(
        grid,
        extent=[x_edges[0], x_edges[-1], y_edges[-1], y_edges[0]],
        origin="upper",
        aspect="auto",
        cmap="YlOrRd",
        vmin=0,
        vmax=vmax,
        alpha=0.88,
    )
    draw_half_pitch(ax)
    ax.set_title(title, fontsize=13)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if stat == "sum":
        cbar.set_label("Total xG in grid cell")
    elif stat == "mean":
        cbar.set_label("Average xG per shot in grid cell")
    else:
        cbar.set_label("Shot count in grid cell")
    plt.tight_layout()
    plt.savefig(filename, dpi=220, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {filename}")



def plot_difference_heatmap(df_for: pd.DataFrame, df_against: pd.DataFrame, title: str, filename: Path):
    grid_for, x_edges, y_edges = compute_grid(df_for, stat="sum")
    grid_against, _, _ = compute_grid(df_against, stat="sum")
    diff = grid_for - grid_against
    vmax = float(np.abs(diff).max()) if np.isfinite(np.abs(diff).max()) else 1.0
    if vmax <= 0:
        vmax = 1.0

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(
        diff,
        extent=[x_edges[0], x_edges[-1], y_edges[-1], y_edges[0]],
        origin="upper",
        aspect="auto",
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
        alpha=0.9,
    )
    draw_half_pitch(ax)
    ax.set_title(title, fontsize=13)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("xG For minus xG Against")
    plt.tight_layout()
    plt.savefig(filename, dpi=220, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {filename}")



def shot_chart_by_result(df: pd.DataFrame, title: str, filename: Path):
    fig, ax = plt.subplots(figsize=(8, 7))
    draw_half_pitch(ax)

    for result in RESULT_ORDER:
        subset = df[df["ResultClean"] == result]
        if subset.empty:
            continue
        style = RESULT_STYLES[result]
        ax.scatter(
            subset["X"],
            subset["Y"],
            s=style["size"],
            marker=style["marker"],
            c=style["color"],
            label=f"{result} ({len(subset)})",
            alpha=0.85,
            edgecolors="black" if style["marker"] != "x" else None,
            linewidths=0.5,
        )

    ax.set_title(title, fontsize=13)
    ax.legend(loc="upper right", frameon=True)
    plt.tight_layout()
    plt.savefig(filename, dpi=220, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {filename}")



def split_maps_by_result(df: pd.DataFrame, out_dir: Path, prefix: str):
    for result in RESULT_ORDER:
        subset = df[df["ResultClean"] == result]
        if subset.empty:
            continue
        fig, ax = plt.subplots(figsize=(8, 7))
        draw_half_pitch(ax)
        style = RESULT_STYLES[result]
        ax.scatter(
            subset["X"], subset["Y"],
            s=style["size"], marker=style["marker"], c=style["color"], alpha=0.85,
            edgecolors="black" if style["marker"] != "x" else None,
            linewidths=0.5,
        )
        ax.set_title(f"{prefix} {result} Shot Map ({len(subset)} shots)", fontsize=13)
        plt.tight_layout()
        filename = out_dir / f"{safe_slug(prefix)}_{safe_slug(result)}_shot_map.png"
        plt.savefig(filename, dpi=220, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved: {filename}")



def sortable_game_key(value):
    text = str(value)
    match = re.search(r"(\d+)", text)
    if match:
        return (0, int(match.group(1)), text)
    return (1, text)



def build_match_summary(df: pd.DataFrame, team_name: str) -> pd.DataFrame:
    games = sorted(df["GameId"].dropna().unique(), key=sortable_game_key)
    rows = []
    for game_id in games:
        match_df = df[df["GameId"] == game_id].copy()
        team_for = match_df[match_df["Team"] == team_name].copy()
        team_against = match_df[match_df["Team"] != team_name].copy()
        if team_for.empty and team_against.empty:
            continue
        rows.append(
            {
                "GameId": game_id,
                "shots_for": len(team_for),
                "shots_against": len(team_against),
                "goals_for": int(team_for["goal"].sum()),
                "goals_against": int(team_against["goal"].sum()),
                "xg_for": float(team_for["xG"].sum()),
                "xg_against": float(team_against["xG"].sum()),
                "avg_xg_per_shot_for": float(team_for["xG"].mean()) if len(team_for) else 0.0,
                "avg_xg_per_shot_against": float(team_against["xG"].mean()) if len(team_against) else 0.0,
            }
        )
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out["xg_diff"] = out["xg_for"] - out["xg_against"]
    out["goal_diff"] = out["goals_for"] - out["goals_against"]
    out["finishing_overperformance"] = out["goals_for"] - out["xg_for"]
    out["goalkeeping_defensive_overperformance"] = out["xg_against"] - out["goals_against"]
    return out



def plot_match_xg_summary(match_df: pd.DataFrame, title: str, filename: Path):
    if match_df.empty:
        return
    x = np.arange(len(match_df))
    width = 0.38

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - width / 2, match_df["xg_for"], width=width, label="xG For")
    ax.bar(x + width / 2, match_df["xg_against"], width=width, label="xG Against")
    ax.plot(x, match_df["goals_for"], marker="o", linestyle="-", label="Goals For")
    ax.plot(x, match_df["goals_against"], marker="o", linestyle="--", label="Goals Against")

    ax.set_xticks(x)
    ax.set_xticklabels(match_df["GameId"].astype(str), rotation=45, ha="right")
    ax.set_ylabel("Goals / xG")
    ax.set_title(title)
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=220, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {filename}")



def plot_finishing_overperformance(match_df: pd.DataFrame, title: str, filename: Path):
    if match_df.empty:
        return
    x = np.arange(len(match_df))
    width = 0.38

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - width / 2, match_df["finishing_overperformance"], width=width, label="Goals For - xG For")
    ax.bar(x + width / 2, match_df["goalkeeping_defensive_overperformance"], width=width, label="xG Against - Goals Against")
    ax.axhline(0, color="black", linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(match_df["GameId"].astype(str), rotation=45, ha="right")
    ax.set_ylabel("Over / Underperformance")
    ax.set_title(title)
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=220, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {filename}")



def phase_bucket(value: str) -> str:
    text = str(value).strip().lower()
    if text == "open play":
        return "Open Play"
    if text == "set piece":
        return "Set Piece"
    return "Unknown"



def summarize_phase(df: pd.DataFrame) -> pd.DataFrame:
    temp = df.copy()
    temp["phase_bucket"] = temp["open_play_or_set_piece"].apply(phase_bucket)
    out = temp.groupby("phase_bucket", dropna=False).agg(
        shots=("goal", "size"),
        goals=("goal", "sum"),
        xg=("xG", "sum"),
        avg_xg_per_shot=("xG", "mean"),
    ).reset_index()
    return out.sort_values("phase_bucket")



def plot_phase_summary(phase_df: pd.DataFrame, title: str, filename: Path):
    if phase_df.empty:
        return
    x = np.arange(len(phase_df))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - width / 2, phase_df["shots"], width=width, label="Shots")
    ax.bar(x + width / 2, phase_df["xg"], width=width, label="xG")
    for i, goals in enumerate(phase_df["goals"]):
        ax.text(x[i] + width / 2, phase_df.loc[phase_df.index[i], "xg"] + 0.05, f"G={int(goals)}", ha="center", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(phase_df["phase_bucket"])
    ax.set_title(title)
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=220, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {filename}")



def player_summary(df: pd.DataFrame) -> pd.DataFrame:
    out = df.groupby("Player", dropna=False).agg(
        shots=("goal", "size"),
        goals=("goal", "sum"),
        xg=("xG", "sum"),
        avg_xg_per_shot=("xG", "mean"),
        avg_distance=("shot_distance", "mean"),
        avg_angle=("shot_angle", "mean"),
    ).reset_index()
    out["goals_minus_xg"] = out["goals"] - out["xg"]
    return out.sort_values(["shots", "xg"], ascending=[False, False])



def plot_top_player_profiles(df: pd.DataFrame, player_df: pd.DataFrame, out_dir: Path, team_name: str, n_players: int = 6):
    top_players = player_df.head(n_players)["Player"].tolist()
    for player in top_players:
        subset = df[df["Player"] == player].copy()
        if subset.empty:
            continue
        title = f"{team_name} - {player} Shot Profile"
        filename = out_dir / f"{safe_slug(team_name)}_{safe_slug(player)}_shot_profile.png"
        shot_chart_by_result(subset, title, filename)



def zone_name(x: float, y: float) -> str:
    if y <= 6 and abs(x) <= 10:
        return "6-yard central"
    if y <= 18 and abs(x) <= 10:
        return "Box central"
    if y <= 18 and x < -10:
        return "Box left"
    if y <= 18 and x > 10:
        return "Box right"
    if y > 18 and abs(x) <= 20:
        return "Outside box central"
    if y > 18 and x < -20:
        return "Outside box left"
    if y > 18 and x > 20:
        return "Outside box right"
    if y > 18 and -20 <= x < 0:
        return "Outside box inner-left"
    return "Outside box inner-right"



def zone_summary(df: pd.DataFrame) -> pd.DataFrame:
    temp = df.copy()
    temp["zone"] = [zone_name(x, y) for x, y in zip(temp["X"], temp["Y"])]
    temp["blocked"] = temp["ResultClean"].eq("Blocked").astype(int)
    out = temp.groupby("zone", dropna=False).agg(
        shots=("goal", "size"),
        goals=("goal", "sum"),
        xg=("xG", "sum"),
        avg_xg_per_shot=("xG", "mean"),
        blocked_rate=("blocked", "mean"),
    ).reset_index()
    return out.sort_values(["xg", "shots"], ascending=[False, False])



def plot_zone_xg(zone_df: pd.DataFrame, title: str, filename: Path):
    if zone_df.empty:
        return
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(zone_df["zone"], zone_df["xg"])
    ax.set_title(title)
    ax.set_ylabel("Total xG")
    ax.set_xticklabels(zone_df["zone"], rotation=45, ha="right")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(filename, dpi=220, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {filename}")



def save_story_summary(team_name: str, df_for: pd.DataFrame, df_against: pd.DataFrame, match_df: pd.DataFrame, out_path: Path):
    lines = []
    lines.append(f"Team: {team_name}")
    lines.append(f"Shots For: {len(df_for)}")
    lines.append(f"Goals For: {int(df_for['goal'].sum())}")
    lines.append(f"xG For: {df_for['xG'].sum():.2f}")
    lines.append(f"Shots Against: {len(df_against)}")
    lines.append(f"Goals Against: {int(df_against['goal'].sum())}")
    lines.append(f"xG Against: {df_against['xG'].sum():.2f}")
    lines.append(f"Season xG Diff: {(df_for['xG'].sum() - df_against['xG'].sum()):.2f}")
    lines.append(f"Finishing Overperformance: {(df_for['goal'].sum() - df_for['xG'].sum()):.2f}")
    lines.append(f"Defensive / Goalkeeping Overperformance: {(df_against['xG'].sum() - df_against['goal'].sum()):.2f}")
    if not match_df.empty:
        lines.append("")
        lines.append("Per-match summary:")
        for _, row in match_df.iterrows():
            lines.append(
                f"Game {row['GameId']}: xG For {row['xg_for']:.2f}, xG Against {row['xg_against']:.2f}, "
                f"Goals {int(row['goals_for'])}-{int(row['goals_against'])}"
            )
    out_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"Saved: {out_path}")



def main():
    df = load_data(CSV_PATH)
    df, model_result = add_xg(df)

    if model_result is not None:
        print("\nFitted local xG model from shot_distance + shot_angle.\n")
        print(model_result.summary())
    else:
        print("\nUsing existing xG column from the CSV.\n")

    team_name = choose_team(df)
    team_slug = safe_slug(team_name)
    out_dir = CSV_PATH.parent / f"{team_slug}_story_outputs"
    out_dir.mkdir(exist_ok=True)

    df_for = df[df["Team"] == team_name].copy()
    df_against = df[df["Team"] != team_name].copy()

    print(f"\nTeam selected: {team_name}")
    print(f"Shots For: {len(df_for)} | xG For: {df_for['xG'].sum():.2f}")
    print(f"Shots Against: {len(df_against)} | xG Against: {df_against['xG'].sum():.2f}")
    print(f"Outputs folder: {out_dir}\n")

    # Heatmaps
    plot_heatmap(df_for, f"{team_name} xG For Heatmap", out_dir / f"{team_slug}_xg_for_heatmap.png", stat=HEATMAP_STAT)
    plot_heatmap(df_against, f"{team_name} xG Against Heatmap", out_dir / f"{team_slug}_xg_against_heatmap.png", stat=HEATMAP_STAT)
    plot_heatmap(pd.concat([df_for, df_against], ignore_index=True), f"{team_name} Combined Heatmap", out_dir / f"{team_slug}_combined_heatmap.png", stat=HEATMAP_STAT)
    plot_difference_heatmap(df_for, df_against, f"{team_name} xG Difference Heatmap", out_dir / f"{team_slug}_xg_difference_heatmap.png")

    # Shot charts by result
    shot_chart_by_result(df_for, f"{team_name} Shot Chart by Result", out_dir / f"{team_slug}_shot_chart_by_result.png")
    split_maps_by_result(df_for, out_dir, f"{team_name}")

    # Match-level story
    match_df = build_match_summary(df, team_name)
    match_csv = out_dir / f"{team_slug}_match_summary.csv"
    match_df.to_csv(match_csv, index=False)
    print(f"Saved: {match_csv}")
    plot_match_xg_summary(match_df, f"{team_name} xG For vs xG Against by Match", out_dir / f"{team_slug}_xg_by_match.png")
    plot_finishing_overperformance(match_df, f"{team_name} Finishing / Goalkeeping Overperformance by Match", out_dir / f"{team_slug}_overperformance_by_match.png")

    # Open play vs set piece
    phase_df = summarize_phase(df_for)
    phase_csv = out_dir / f"{team_slug}_phase_summary.csv"
    phase_df.to_csv(phase_csv, index=False)
    print(f"Saved: {phase_csv}")
    plot_phase_summary(phase_df, f"{team_name} Open Play vs Set Piece", out_dir / f"{team_slug}_open_play_vs_set_piece.png")

    # Player profiles
    player_df = player_summary(df_for)
    player_csv = out_dir / f"{team_slug}_player_summary.csv"
    player_df.to_csv(player_csv, index=False)
    print(f"Saved: {player_csv}")
    plot_top_player_profiles(df_for, player_df, out_dir, team_name, n_players=TOP_N_PLAYERS)

    # Zone summaries
    zone_for_df = zone_summary(df_for)
    zone_against_df = zone_summary(df_against)
    zone_for_csv = out_dir / f"{team_slug}_zone_summary_for.csv"
    zone_against_csv = out_dir / f"{team_slug}_zone_summary_against.csv"
    zone_for_df.to_csv(zone_for_csv, index=False)
    zone_against_df.to_csv(zone_against_csv, index=False)
    print(f"Saved: {zone_for_csv}")
    print(f"Saved: {zone_against_csv}")
    plot_zone_xg(zone_for_df, f"{team_name} xG by Zone (For)", out_dir / f"{team_slug}_zone_xg_for.png")
    plot_zone_xg(zone_against_df, f"{team_name} xG by Zone (Against)", out_dir / f"{team_slug}_zone_xg_against.png")

    # Text summary
    save_story_summary(team_name, df_for, df_against, match_df, out_dir / f"{team_slug}_story_summary.txt")

    print("\nDone. The folder now has heatmaps, shot charts, match summaries, player profiles, and zone tables.")


if __name__ == "__main__":
    main()



Fitted local xG model from shot_distance + shot_angle.

                           Logit Regression Results                           
Dep. Variable:                   goal   No. Observations:                  385
Model:                          Logit   Df Residuals:                      382
Method:                           MLE   Df Model:                            2
Date:                Mon, 09 Mar 2026   Pseudo R-squ.:                  0.1472
Time:                        21:29:44   Log-Likelihood:                -166.64
converged:                       True   LL-Null:                       -195.40
Covariance Type:            nonrobust   LLR p-value:                 3.236e-13
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.2855      0.678      0.421      0.674      -1.044       1.615
shot_distance    -0.1210      0.031     -3.851      0.000      -0

C:\Users\User\AppData\Local\Temp\ipykernel_170956\3182571160.py:497: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(zone_df["zone"], rotation=45, ha="right")


Saved: C:\Users\User\OneDrive - 2020 Companies\Desktop\IHS Soccer\Boys\2026\xG Model\ingraham_story_outputs\ingraham_zone_xg_for.png


C:\Users\User\AppData\Local\Temp\ipykernel_170956\3182571160.py:497: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(zone_df["zone"], rotation=45, ha="right")


Saved: C:\Users\User\OneDrive - 2020 Companies\Desktop\IHS Soccer\Boys\2026\xG Model\ingraham_story_outputs\ingraham_zone_xg_against.png
Saved: C:\Users\User\OneDrive - 2020 Companies\Desktop\IHS Soccer\Boys\2026\xG Model\ingraham_story_outputs\ingraham_story_summary.txt

Done. The folder now has heatmaps, shot charts, match summaries, player profiles, and zone tables.
